# 04 — The key libraries and their APIs

The whole system stands on four third-party libraries plus `asyncio` from the
standard library. This notebook shows each one's API **the way this project
actually uses it** — with runnable examples where no network is needed.

| library | role here | hot path? |
|---|---|---|
| `river` | online machine learning (learn one sample at a time) | yes |
| `websockets` | raw WebSocket client for the market-data stream | yes |
| `duckdb` | embedded analytics database (the "cold store") | no (background writer) |
| `alpaca-py` | REST client for orders + historical data | no (order path runs in threads) |
| `asyncio` | concurrency: queues, tasks, backpressure | yes (the skeleton) |

## 1. River — online machine learning

**The big idea:** classic ML (scikit-learn) is *batch* — you collect a dataset,
call `fit(X, y)` once, then `predict()`. River inverts this: models start empty
and learn **one observation at a time**, forever:

```
model.predict_one(x)   # x is a plain dict: {"feature_name": value}
model.learn_one(x, y)  # update weights/tree with a single example
```

Why dicts instead of numpy arrays? Features can appear/disappear per sample
(sparse-friendly), there's no fitted "schema", and there's no pandas/numpy
dependency in the update path — which is exactly what a µs-budget loop wants.

Everything stays O(1) per sample: a `LinearRegression` does one SGD step; a
`HoeffdingTreeRegressor` updates sufficient statistics and only *splits* a leaf
when a statistical test (the Hoeffding bound) says one feature is reliably
better — that's how a decision tree becomes incremental.

In [1]:
from river import linear_model, tree
import numpy as np

rng = np.random.default_rng(0)

lin = linear_model.LinearRegression()      # empty model, knows nothing
hoeff = tree.HoeffdingTreeRegressor(grace_period=200)

for _ in range(2000):
    a, b = rng.normal(), rng.normal()
    x, y = {"a": a, "b": b}, 2 * a - b     # the relationship to discover
    lin.learn_one(x, y)
    hoeff.learn_one(x, y)

probe = {"a": 1.0, "b": 0.0}               # true answer: 2.0
print("linear   :", round(lin.predict_one(probe), 3))
print("hoeffding:", round(hoeff.predict_one(probe), 3))
print("linear weights:", {k: round(v, 3) for k, v in lin.weights.items()})

linear   : 2.0
hoeffding: 2.001
linear weights: {'a': 2.0, 'b': -1.0}


Classifiers add one method — `predict_proba_one` — which returns a dict of
class probabilities. Our `kind="classifier"` model uses a 3-class
`HoeffdingTreeClassifier` (down-through-band / dead-zone / up-through-band) and
converts probabilities back to an expected return:

In [2]:
cls = tree.HoeffdingTreeClassifier(grace_period=100)
for _ in range(2000):
    a = rng.choice([-1.0, 0.0, 1.0])
    label = int(a)                          # class IS the sign of the driver
    cls.learn_one({"a": a + rng.normal(0, 0.2)}, label)

proba = cls.predict_proba_one({"a": 0.9})
print({k: round(v, 3) for k, v in proba.items()})
# our wrapper: expected_return ~ (P(+1) - P(-1)) * typical_tail_move

{-1: 0.0, 0: 0.003, 1: 0.997}


Where it lives in this repo: `src/signals/model/online.py` wraps River behind
`OnlineModel` (kinds: `linear`, `hoeffding`, `classifier`, `meta`) so the rest
of the system never imports River directly.

## 2. websockets — the market-data pipe

A WebSocket is a persistent two-way TCP connection speaking a standard framing
protocol — the norm for streaming market data. The `websockets` library's whole
client API is three ideas:

```python
async with connect(url, ssl=ctx, ping_interval=20) as ws:   # 1. connect
    await ws.send(json.dumps({"action": "auth", ...}))       # 2. send
    async for raw in ws:                                     # 3. iterate frames
        handle(json.loads(raw))
```

- `ping_interval=20` is the **heartbeat**: the library pings every 20s; a peer
  that stops answering raises `ConnectionClosed`, our loop catches it and
  reconnects with exponential backoff (`src/signals/data/alpaca.py`).
- We drive `websockets` directly instead of using alpaca-py's stream wrapper so
  reconnect policy, timestamp parsing, and per-message cost stay under our
  control.

Runnable miniature — a client and server in one kernel:

In [3]:
import asyncio, json
from websockets.asyncio.client import connect
from websockets.asyncio.server import serve

async def tiny_server(ws):
    await ws.send(json.dumps([{"T": "success", "msg": "connected"}]))
    async for raw in ws:                       # echo trades back
        msg = json.loads(raw)
        await ws.send(json.dumps([{"T": "t", "S": msg["symbol"], "p": 100.0}]))

async def demo():
    async with serve(tiny_server, "127.0.0.1", 0) as server:
        port = server.sockets[0].getsockname()[1]
        async with connect(f"ws://127.0.0.1:{port}") as ws:
            print("banner:", await ws.recv())
            await ws.send(json.dumps({"symbol": "SPY"}))
            print("frame :", await ws.recv())

await demo()   # top-level await works in notebooks

banner: [{"T": "success", "msg": "connected"}]
frame : [{"T": "t", "S": "SPY", "p": 100.0}]


## 3. DuckDB — the cold store

DuckDB is "SQLite for analytics": a full SQL database living in one file, no
server process, but **columnar** — scans and aggregations over millions of rows
run at vectorized speed. That's why 3.9M-event session files stay pleasant to
query.

API in one breath: `connect(path)` → `.execute(sql, params)` →
`.fetchall()` / `.fetchone()` / **`.df()`** (straight to pandas). Batch inserts
use `executemany`. Two rules this project lives by:

- **Single writer.** One process holds the write lock; a crashed writer leaves
  a `.wal` file (write-ahead log) that is replayed on the next connect.
- `read_only=True` for analysis connections — and never any read in the hot
  loop; the pipeline only ever *appends*, in batches, from a background task
  (`src/signals/storage/coldstore.py`).

In [4]:
from pathlib import Path
import duckdb

DB = Path("..") / "data" / "session.duckdb"
conn = duckdb.connect(str(DB), read_only=True)
df = conn.execute('''
    SELECT symbol,
           count(*)                                    AS quotes,
           round(avg((ask - bid) / ((ask + bid)/2)) * 1e4, 2) AS avg_spread_bps,
           round((max(ts_ns) - min(ts_ns)) / 60e9, 1)  AS span_min
    FROM quotes GROUP BY symbol ORDER BY quotes DESC
''').df()
conn.close()
df

,symbol,quotes,avg_spread_bps,span_min
0,BTC/USD,4049,11.47,29.9
1,ETH/USD,275,11.58,29.8


## 4. alpaca-py — orders and history (cold path only)

alpaca-py is the official SDK. We use exactly two corners of it, always inside
`asyncio.to_thread` because it's synchronous:

```python
# Orders — the ONLY door to the market, guarded to paper in PaperExecutor:
from alpaca.trading.client import TradingClient
from alpaca.trading.requests import MarketOrderRequest
from alpaca.trading.enums import OrderSide, TimeInForce

client = TradingClient(api_key, secret_key, paper=True)      # paper=True or refuse
order = client.submit_order(MarketOrderRequest(
    symbol="BTC/USD", qty=0.001,
    side=OrderSide.BUY, time_in_force=TimeInForce.GTC))
client.get_order_by_id(order.id)      # poll fill status
client.get_all_positions()            # what we actually hold
client.close_all_positions(cancel_orders=True)   # the kill switch

# History — warm-up bars / research:
from alpaca.data.historical import CryptoHistoricalDataClient
from alpaca.data.requests import CryptoBarsRequest
from alpaca.data.timeframe import TimeFrame
```

(Not executed here — it needs keys and network; see
`scripts/paper_order_check.py` for the full round-trip.) The **streaming**
endpoints we talk to with raw `websockets` instead; the JSON message shapes
(`{"T": "q", "bp": ..., "ap": ...}`) are parsed in
`src/signals/data/alpaca.py::parse_message`.

## 5. asyncio — the skeleton

One thread, one event loop, many concurrent tasks that yield at every `await`.
The pipeline is a chain of tasks connected by **bounded queues**:

- `asyncio.Queue(maxsize=N)` — `await put()` **blocks when full**. That's
  *backpressure*: a slow consumer automatically slows the producer instead of
  growing memory forever.
- `put_nowait()` + catch `QueueFull` — the *non-blocking* variant used for the
  logging tap: losing a log row is fine, stalling the trading loop is not.
- `asyncio.to_thread(fn)` — runs blocking code (alpaca-py, DuckDB writes) in a
  worker thread so the loop never stops.

Miniature of the exact pattern:

In [5]:
import asyncio

async def producer(q):
    dropped = 0
    for i in range(2000):
        try:
            q.put_nowait(i)          # logging-tap style: never block
        except asyncio.QueueFull:
            dropped += 1
    return dropped

async def slow_consumer(q):
    seen = 0
    while True:
        await q.get()
        seen += 1
        if seen % 100 == 0:
            await asyncio.sleep(0)   # pretend to do work
        if q.empty():
            return seen

q = asyncio.Queue(maxsize=500)
dropped = await producer(q)
seen = await slow_consumer(q)
print(f"queue cap 500: consumer saw {seen}, producer dropped {dropped} (counted, not fatal)")

queue cap 500: consumer saw 500, producer dropped 1500 (counted, not fatal)


Full wiring: `src/signals/pipeline.py` — ingest task → consumer task (the
decision loop) → fire-and-forget order tasks → cold-store writer task, all
joined by queues, all cancellable for graceful shutdown.

Next: [05_jargon_glossary](05_jargon_glossary.ipynb) for every term this
project throws around.